# Demand Forecasting
## AeroNet Lite - Module 5 (Part A)
**BS Data Science AI Semester Project - SP2026**

This notebook covers Part A of the ML Pipeline: training regression models to forecast
delivery demand from the Amazon Delivery Dataset. The output of this notebook feeds directly
into the shared grid model used by all other modules.

---
### What this notebook does
1. Loads and cleans the Amazon Delivery Dataset
2. Engineers time, distance, and categorical features
3. Trains Linear Regression and Random Forest Regressor
4. Evaluates with MAE and RMSE, plots model comparison and feature importance
5. Exposes `get_demand_forecast()` and `DEMAND_BY_AREA` for teammate integration

### Integration points exposed to teammates
| Function / Variable | Used by | Purpose |
|---|---|---|
| `get_demand_forecast(hour, weather, traffic, area)` | Module 2 (Fleet Selector), main.py | Returns predicted delivery time as demand proxy |
| `DEMAND_BY_AREA` | Module 1 (Grid Model), main.py | Dict mapping zone type to avg demand for grid cell init |


## 0. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
np.random.seed(42)

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('All libraries loaded.')
print('demand_forecasting.ipynb - Module 5 Part A')


## A1. Load Dataset

In [ ]:
# Update this path to wherever the CSV lives in your project folder
DATA_PATH = 'amazon_delivery.csv'

df_raw = pd.read_csv(DATA_PATH)

print('Shape:', df_raw.shape)
print('\nColumns:', df_raw.columns.tolist())
print('\nFirst 5 rows:')
df_raw.head()


In [ ]:
print('Data types:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())


## A2. Data Cleaning

In [ ]:
df = df_raw.copy()

# Strip whitespace from string columns (Traffic and Area have trailing spaces)
str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].str.strip()

# Fill missing Weather with mode
weather_mode = df['Weather'].mode()[0]
df['Weather'] = df['Weather'].fillna(weather_mode)
print(f'Weather nulls filled with mode: "{weather_mode}"')

# Fill missing Agent_Rating with median
rating_median = df['Agent_Rating'].median()
df['Agent_Rating'] = df['Agent_Rating'].fillna(rating_median)
print(f'Agent_Rating nulls filled with median: {rating_median}')

# Remove the string 'NaN' in Traffic (it is a string, not a real NaN)
df = df[df['Traffic'] != 'NaN']
print(f'Rows after removing Traffic=NaN: {len(df)}')

# Parse dates to extract day of week and month
df['Order_Date'] = pd.to_datetime(df['Order_Date'], dayfirst=True, errors='coerce')
df['Day_of_Week'] = df['Order_Date'].dt.dayofweek   # 0=Monday
df['Month']       = df['Order_Date'].dt.month

# Parse order time to extract hour
df['Order_Hour'] = pd.to_datetime(
    df['Order_Time'], format='%H:%M:%S', errors='coerce'
).dt.hour

# Compute haversine distance (km) between store and drop location
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi    = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['Distance_km'] = haversine(
    df['Store_Latitude'], df['Store_Longitude'],
    df['Drop_Latitude'],  df['Drop_Longitude']
)

print('\nCleaning complete.')
print('Unique Weather:', df['Weather'].unique())
print('Unique Traffic:', df['Traffic'].unique())
print('Unique Area:   ', df['Area'].unique())


## A3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Target distribution
axes[0, 0].hist(df['Delivery_Time'], bins=40, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Delivery Time Distribution (minutes)')
axes[0, 0].set_xlabel('Delivery Time')
axes[0, 0].set_ylabel('Count')

# Delivery time by traffic
traffic_order = ['Low', 'Medium', 'High', 'Jam']
traffic_data  = [df[df['Traffic'] == t]['Delivery_Time'].values for t in traffic_order]
axes[0, 1].boxplot(traffic_data, labels=traffic_order)
axes[0, 1].set_title('Delivery Time by Traffic')
axes[0, 1].set_ylabel('Delivery Time (min)')

# Delivery time by weather
weather_avg = df.groupby('Weather')['Delivery_Time'].mean().sort_values(ascending=False)
axes[0, 2].bar(weather_avg.index, weather_avg.values, color='coral', edgecolor='white')
axes[0, 2].set_title('Avg Delivery Time by Weather')
axes[0, 2].set_ylabel('Avg Delivery Time (min)')
axes[0, 2].tick_params(axis='x', rotation=30)

# Delivery time by area
area_avg = df.groupby('Area')['Delivery_Time'].mean().sort_values(ascending=False)
axes[1, 0].bar(area_avg.index, area_avg.values, color='mediumseagreen', edgecolor='white')
axes[1, 0].set_title('Avg Delivery Time by Area')
axes[1, 0].set_ylabel('Avg Delivery Time (min)')
axes[1, 0].tick_params(axis='x', rotation=15)

# Distance vs Delivery Time
sample = df.sample(2000, random_state=42)
axes[1, 1].scatter(sample['Distance_km'], sample['Delivery_Time'],
                   alpha=0.3, s=10, color='orchid')
axes[1, 1].set_title('Distance vs Delivery Time')
axes[1, 1].set_xlabel('Distance (km)')
axes[1, 1].set_ylabel('Delivery Time (min)')

# Hourly demand pattern
hour_avg = df.groupby('Order_Hour')['Delivery_Time'].mean()
axes[1, 2].plot(hour_avg.index, hour_avg.values, marker='o', color='darkorange')
axes[1, 2].set_title('Avg Delivery Time by Hour of Day')
axes[1, 2].set_xlabel('Hour')
axes[1, 2].set_ylabel('Avg Delivery Time (min)')

plt.tight_layout()
plt.suptitle('Exploratory Data Analysis - Amazon Delivery Dataset', y=1.01, fontsize=14)
plt.show()


## A4. Feature Engineering

In [ ]:
le_weather = LabelEncoder()
le_traffic = LabelEncoder()
le_area    = LabelEncoder()
le_vehicle = LabelEncoder()

df['Weather_enc'] = le_weather.fit_transform(df['Weather'])
df['Traffic_enc'] = le_traffic.fit_transform(df['Traffic'])
df['Area_enc']    = le_area.fit_transform(df['Area'])
df['Vehicle_enc'] = le_vehicle.fit_transform(df['Vehicle'])

print('Weather classes:', dict(zip(le_weather.classes_, le_weather.transform(le_weather.classes_))))
print('Traffic classes:', dict(zip(le_traffic.classes_, le_traffic.transform(le_traffic.classes_))))
print('Area classes:   ', dict(zip(le_area.classes_,    le_area.transform(le_area.classes_))))

REGRESSION_FEATURES = [
    'Agent_Age', 'Agent_Rating',
    'Distance_km',
    'Order_Hour', 'Day_of_Week', 'Month',
    'Weather_enc', 'Traffic_enc', 'Area_enc', 'Vehicle_enc'
]
TARGET = 'Delivery_Time'

df_model = df[REGRESSION_FEATURES + [TARGET]].dropna()
print(f'\nFinal dataset for regression: {df_model.shape[0]} rows, {len(REGRESSION_FEATURES)} features')
df_model[REGRESSION_FEATURES].describe().round(2)


## A5. Train / Test Split

In [ ]:
X = df_model[REGRESSION_FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set : {X_train.shape[0]} samples')
print(f'Test set     : {X_test.shape[0]} samples')


## A6. Model 1 - Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr  = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))

print('Linear Regression Results')
print(f'  MAE  : {mae_lr:.2f} minutes')
print(f'  RMSE : {rmse_lr:.2f} minutes')

coef_df = pd.DataFrame({'Feature': REGRESSION_FEATURES, 'Coefficient': lr.coef_})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print('\nTop feature coefficients:')
print(coef_df.to_string(index=False))


## A7. Model 2 - Random Forest Regressor

In [ ]:
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)

y_pred_rf = rf_reg.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print('Random Forest Regressor Results')
print(f'  MAE  : {mae_rf:.2f} minutes')
print(f'  RMSE : {rmse_rf:.2f} minutes')

fi_df = pd.DataFrame({'Feature': REGRESSION_FEATURES, 'Importance': rf_reg.feature_importances_})
fi_df = fi_df.sort_values('Importance', ascending=False)
print('\nFeature importances:')
print(fi_df.to_string(index=False))


## A8. Model Comparison and Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = ['Linear Regression', 'Random Forest']
maes   = [mae_lr, mae_rf]
rmses  = [rmse_lr, rmse_rf]

x = np.arange(len(models))
w = 0.35
axes[0].bar(x - w/2, maes,  w, label='MAE',  color='steelblue')
axes[0].bar(x + w/2, rmses, w, label='RMSE', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_ylabel('Error (minutes)')
axes[0].set_title('Regression Model Comparison')
axes[0].legend()
for i, (m, r) in enumerate(zip(maes, rmses)):
    axes[0].text(i - w/2, m + 0.3, f'{m:.1f}', ha='center', fontsize=9)
    axes[0].text(i + w/2, r + 0.3, f'{r:.1f}', ha='center', fontsize=9)

lims = [min(y_test.min(), y_pred_lr.min()), max(y_test.max(), y_pred_lr.max())]

axes[1].scatter(y_test[:500], y_pred_lr[:500], alpha=0.4, s=10, color='steelblue')
axes[1].plot(lims, lims, 'r--', linewidth=1)
axes[1].set_xlabel('Actual Delivery Time')
axes[1].set_ylabel('Predicted Delivery Time')
axes[1].set_title('Linear Regression: Actual vs Predicted')

axes[2].scatter(y_test[:500], y_pred_rf[:500], alpha=0.4, s=10, color='mediumseagreen')
axes[2].plot(lims, lims, 'r--', linewidth=1)
axes[2].set_xlabel('Actual Delivery Time')
axes[2].set_ylabel('Predicted Delivery Time')
axes[2].set_title('Random Forest: Actual vs Predicted')

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(fi_df['Feature'], fi_df['Importance'], color='darkorange')
ax.set_xlabel('Importance Score')
ax.set_title('Random Forest - Feature Importance for Demand Forecasting')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## A8b. Demand Heatmap by Hour and Area

In [ ]:
# Pivot: rows = area, columns = order hour, values = avg delivery time
# This becomes the demand heatmap referenced in Section 10 of the project doc

pivot = df.pivot_table(
    values='Delivery_Time',
    index='Area',
    columns='Order_Hour',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.4, annot=False, ax=ax)
ax.set_title('Demand Heatmap - Avg Delivery Time by Area and Hour of Day')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Area Type')
plt.tight_layout()
plt.show()

print('Heatmap shows peak demand periods per zone type.')
print('Use this to decide when to dispatch additional drones in the simulation.')


## A9. Best Model Selection

In [ ]:
best_model_name = 'Random Forest' if mae_rf < mae_lr else 'Linear Regression'
best_model      = rf_reg if mae_rf < mae_lr else lr
best_mae        = min(mae_rf, mae_lr)
best_rmse       = min(rmse_rf, rmse_lr)

print('Demand Forecasting Summary')
print('=' * 40)
print(f'Best model : {best_model_name}')
print(f'MAE        : {best_mae:.2f} minutes')
print(f'RMSE       : {best_rmse:.2f} minutes')
print()
print('Interpretation:')
print(f'  On average the model predicts delivery time within {best_mae:.1f} minutes.')
print('  This forecast is used in the simulation to estimate demand per grid zone.')
print('  Zones with higher predicted delivery times carry higher demand values.')


## A10. Demand Mapping to Grid Zones

In [ ]:
# Average delivery time per area type used as a demand proxy for grid cells.
# Teammate integration: import DEMAND_BY_AREA from this notebook or from ml_pipeline.py

area_demand  = df.groupby('Area')['Delivery_Time'].mean().round(2)
DEMAND_BY_AREA = area_demand.to_dict()

print('DEMAND_BY_AREA (average delivery time in minutes, used as demand proxy):')
for area, val in DEMAND_BY_AREA.items():
    print(f'  {area:20s} : {val:.2f} min')


## A11. Integration Function - get_demand_forecast()

In [ ]:
def get_demand_forecast(
    hour,
    weather,
    traffic,
    area,
    agent_age=30,
    agent_rating=4.5,
    distance_km=5.0,
    day_of_week=0,
    month=6,
    vehicle='motorcycle'
):
    """
    Predict delivery demand (as delivery-time proxy) given contextual inputs.

    Parameters
    ----------
    hour         : int   - Hour of the day (0-23)
    weather      : str   - One of: Sunny, Cloudy, Stormy, Fog, Windy, Sandstorms
    traffic      : str   - One of: Low, Medium, High, Jam
    area         : str   - One of: Urban, Metropolitan, Semi-Urban, Other
    agent_age    : int   - Default 30
    agent_rating : float - Default 4.5
    distance_km  : float - Estimated delivery distance in km
    day_of_week  : int   - 0=Monday, 6=Sunday
    month        : int   - 1-12
    vehicle      : str   - Vehicle type

    Returns
    -------
    float : Predicted delivery time in minutes (demand proxy)
    """
    try:
        w_enc = le_weather.transform([weather])[0]
    except ValueError:
        w_enc = 0
    try:
        t_enc = le_traffic.transform([traffic])[0]
    except ValueError:
        t_enc = 1
    try:
        a_enc = le_area.transform([area])[0]
    except ValueError:
        a_enc = 0
    try:
        v_enc = le_vehicle.transform([vehicle])[0]
    except ValueError:
        v_enc = 0

    features = np.array([[
        agent_age, agent_rating, distance_km,
        hour, day_of_week, month,
        w_enc, t_enc, a_enc, v_enc
    ]])
    return float(best_model.predict(features)[0])


# Quick smoke tests
print('get_demand_forecast() smoke tests:')
print(f'  Sunny, Low, Urban, 8am    -> {get_demand_forecast(8, "Sunny", "Low", "Urban"):.1f} min')
print(f'  Stormy, Jam, Metro, 5pm   -> {get_demand_forecast(17, "Stormy", "Jam", "Metropolitian"):.1f} min')
print(f'  Fog, Medium, Semi-Urban, 12pm -> {get_demand_forecast(12, "Fog", "Medium", "Semi-Urban"):.1f} min')


## A12. Integration with Other Modules

This section shows how teammates call into this notebook's outputs.
**No changes to this notebook are required after initial run.**


In [ ]:
# -----------------------------------------------------------------------
# HOW MODULE 2 (Fleet Selector) calls this notebook
# -----------------------------------------------------------------------
# In fleet_selector.py or fleet_selector.ipynb:
#
#   from ml_pipeline import get_demand_forecast, DEMAND_BY_AREA
#
#   # Estimate peak demand for each zone before choosing drone count
#   urban_demand = get_demand_forecast(hour=17, weather='Sunny',
#                                      traffic='High', area='Urban')
#   if urban_demand > 130:
#       recommended_extra_drones = 1
#   else:
#       recommended_extra_drones = 0

print('--- Module 2 (Fleet Selector) integration example ---')
urban_peak   = get_demand_forecast(17, 'Sunny', 'High', 'Urban')
metro_peak   = get_demand_forecast(17, 'Sunny', 'Jam',  'Metropolitian')
print(f'  Urban peak demand (5pm)      : {urban_peak:.1f} min')
print(f'  Metropolitan peak demand (5pm): {metro_peak:.1f} min')
if urban_peak > 130:
    print('  Fleet recommendation: add 1 extra light drone for Urban zone.')
else:
    print('  Fleet recommendation: current fleet is sufficient.')


In [ ]:
# -----------------------------------------------------------------------
# HOW MODULE 1 (Grid Model) uses DEMAND_BY_AREA
# -----------------------------------------------------------------------
# In grid_model.py:
#
#   from ml_pipeline import DEMAND_BY_AREA
#
#   for cell in grid.cells:
#       cell['demand'] = DEMAND_BY_AREA.get(cell['zone'], 120)

print('--- Module 1 (Grid Model) integration example ---')
print('DEMAND_BY_AREA values that will initialize grid cell demand fields:')
for zone, val in DEMAND_BY_AREA.items():
    print(f'  grid cell zone="{zone}" -> demand set to {val:.1f}')


In [ ]:
# -----------------------------------------------------------------------
# HOW main.py calls this at simulation steps 15-17
# -----------------------------------------------------------------------
print('--- main.py simulation steps 15-17 ---')

# Step 15
forecast_15 = get_demand_forecast(hour=15, weather='Cloudy', traffic='Medium', area='Urban')
print(f'Step 15: Demand forecast computed -> {forecast_15:.1f} min estimated delivery time')
if forecast_15 > 130:
    print('         High demand detected. Recommending 1 additional delivery dispatch.')
else:
    print('         Moderate demand. Current fleet sufficient.')

# Step 16
forecast_16 = get_demand_forecast(hour=16, weather='Sunny', traffic='High', area='Metropolitian')
print(f'Step 16: Metropolitan zone forecast -> {forecast_16:.1f} min. Adjusting drone priority.')

# Step 17: could include cross-val here; that is kept in anomaly_classifier.ipynb
print('Step 17: Demand module complete. Anomaly detection handled in anomaly_classifier.ipynb.')


## Summary

In [ ]:
print('DEMAND FORECASTING NOTEBOOK - SUMMARY')
print('=' * 55)
print(f'Dataset      : Amazon Delivery Dataset ({df_model.shape[0]} rows)')
print(f'Features     : {REGRESSION_FEATURES}')
print(f'Best model   : {best_model_name}')
print(f'MAE          : {best_mae:.2f} minutes')
print(f'RMSE         : {best_rmse:.2f} minutes')
print()
print('Integration exports:')
print('  get_demand_forecast(hour, weather, traffic, area, ...) -> float')
print('  DEMAND_BY_AREA -> dict  (area_type -> avg delivery time)')
print()
print('Simulation steps handled here: 15, 16, (17 cross-val in anomaly notebook)')
print('=' * 55)
